# Sequence lab 1 · Explain the RNN forecaster you built: which past time steps does it use?

**Author:** Xiangqi Kong

**Follows:** the earlier lab *Processing sequences using RNNs and CNNs* (same data, same task, same models, now in PyTorch) · **Decks:** Post-hoc Model-Specific (*Integrated Gradients*), Post-hoc Model-Agnostic (*Occlusion*) · **Runtime:** about 2 minutes on CPU · **GPU:** not needed

In the earlier lab you trained networks that read **50 time steps** of a wavy series and predict the **next value**. You compared their errors. You never asked *how* they make the forecast.

This lab asks one question: **which of the 50 past steps does the model use?**

1. Rebuild two of the forecasters: the **linear** baseline and the **LSTM**.
2. Write **Integrated Gradients (IG)** yourself and check it on the linear model, where the right answer is known exactly.
3. Read the IG explanation of the LSTM: one forecast, then the average over many.
4. See how the **baseline** changes the explanation.
5. Write **occlusion over time windows** (a model-agnostic method) and compare it with IG.
6. **Test the explanation:** if IG says only the last few steps matter, hiding all the others should not hurt. Does it?

Before you run anything, write down your guess: *how many of the 50 steps does the LSTM really need?*

> **Easy version.** Almost all the code is written. Each **✏️ TODO** has exactly **one blank, written `BLANK`**, and the comment above it
> says what goes there. Replace `BLANK`, run the cell, then run the **✅ check** cell after it. If you see
> `NameError: name 'BLANK' is not defined`, a blank is still empty. The `_walkthrough` notebook in this folder explains every line; the `_solution` notebook has all the answers.

> **In the lab session: about 20 minutes.** Do the sections marked 🟢 core in order; they are what we discuss at the end.
> Sections marked ⚪ if time can be skipped: no 🟢 core section needs them, so come back to them at home.
>
> 🟢 core: 1 data and models, 2 IG by hand (TODO 1), 3 completeness, 4 which time steps, 7 keep the last k steps (TODO 3).  
> ⚪ if time: 5 baselines, 6 occlusion over time windows (TODO 2).

## 0. Setup

> **What this cell does.** It loads the libraries and fixes the random seed. Just run it.

In [ ]:
import time
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import torch, torch.nn as nn
from scipy.stats import spearmanr

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
plt.rcParams.update(
    {"axes.spines.top": False, "axes.spines.right": False, "figure.dpi": 110}
)
BLUE, RED, GREY = "#2563EB", "#DC2626", "#9CA3AF"
print(
    "torch",
    torch.__version__,
    "| the models are small, so this lab runs on the CPU",
)

## 1. The same data and the same models as the earlier lab · 🟢 core

`generate_time_series` is copied from the earlier notebook: every series is **two sine waves plus a little noise**. The model sees steps 0–49 and predicts step 50.

The two models are the earlier lab's *linear baseline* (`Flatten` + `Dense(1)`) and its *deep RNN with LSTM cells* (two layers of 20 units), written in PyTorch.

> **What this cell does.** It makes 10,000 wavy series (the same generator as in your earlier lab), defines the two models (a linear model and an LSTM), trains both for a few seconds, and prints their errors. Just run it.

In [ ]:
def generate_time_series(batch_size, n_steps):
    freq1, freq2, offsets1, offsets2 = np.random.rand(4, batch_size, 1)
    time_ = np.linspace(0, 1, n_steps)
    series = 0.5 * np.sin((time_ - offsets1) * (freq1 * 10 + 10))  #   wave 1
    series += 0.2 * np.sin((time_ - offsets2) * (freq2 * 20 + 20))  # + wave 2
    series += 0.1 * (np.random.rand(batch_size, n_steps) - 0.5)  # + noise
    return series[..., np.newaxis].astype(np.float32)


n_steps = 50
series = torch.tensor(generate_time_series(10000, n_steps + 1))
X_train, y_train = series[:7000, :n_steps], series[:7000, -1, 0]
X_valid, y_valid = series[7000:9000, :n_steps], series[7000:9000, -1, 0]
print(
    "X_train",
    tuple(X_train.shape),
    "(series, time steps, 1 feature)   y_train",
    tuple(y_train.shape),
)


class LinearForecaster(nn.Module):  # Flatten + Dense(1)
    def __init__(self):
        super().__init__()
        self.out = nn.Linear(n_steps, 1)

    def forward(self, x):
        return self.out(x.flatten(1)).squeeze(-1)


# LSTM(20) -> LSTM(20) -> Dense(1) on the last state
class LSTMForecaster(nn.Module):
    def __init__(self, units=20):
        super().__init__()
        self.rnn = nn.LSTM(1, units, num_layers=2, batch_first=True)
        self.out = nn.Linear(units, 1)

    def forward(self, x):
        return self.out(self.rnn(x)[0][:, -1]).squeeze(-1)


def train(model, epochs=20, lr=3e-3, batch=64):
    opt = torch.optim.Adam(model.parameters(), lr)
    t = time.time()
    for _ in range(epochs):
        for b in torch.randperm(len(X_train)).split(batch):
            loss = nn.functional.mse_loss(model(X_train[b]), y_train[b])
            opt.zero_grad()
            loss.backward()
            opt.step()
    model.eval()
    # from now on we only need gradients with respect to the input
    for p in model.parameters():
        p.requires_grad_(False)
    print(f"{type(model).__name__}: trained in {time.time() - t:.0f} s")
    return model


linear, lstm = train(LinearForecaster()), train(LSTMForecaster())
MODELS = {"Linear": linear, "LSTM": lstm}


@torch.no_grad()
def mse(model, X=X_valid, y=y_valid):
    return float(((model(X) - y) ** 2).mean())


display(
    pd.DataFrame(
        {
            "Validation MSE": {
                "Naive (repeat the last value)": float(
                    ((X_valid[:, -1, 0] - y_valid) ** 2).mean()
                ),
                "Linear": mse(linear),
                "LSTM": mse(lstm),
            }
        }
    ).round(4)
)

> **What to notice.** Both models beat the naive forecast (repeat the last value), and the LSTM is the best. These are the models we now explain.

## 2. Integrated Gradients by hand · 🟢 core

IG explains the **difference** between the forecast for the series $x$ and the forecast for a **baseline** series $x'$ (here: a flat line at zero). Each time step $i$ gets

$$\mathrm{IG}_i = (x_i - x'_i) \times \frac{1}{m} \sum_{k=1}^{m} \frac{\partial f}{\partial x_i}\Big(x' + \alpha_k (x - x')\Big), \qquad \alpha_k = \frac{k - 0.5}{m}$$

In words: **walk** from the baseline to the series in $m$ small steps, **average the gradient** along the way, and **multiply** by how far each input moved.

Two facts let us check the code:

- **Linear model.** The gradient is the weight $w_i$ everywhere, so $\mathrm{IG}_i = w_i (x_i - x'_i)$ exactly.
- **Completeness.** For any model the attributions add up to $f(x) - f(x')$.

> **The idea in words.** Start from a *baseline* series that carries no information (a flat line at zero). Walk from the baseline to the real series in small steps. At every step ask the model: *if this input value grew a little, how much would the forecast change?* (that is the gradient). Average these answers along the walk and multiply by how far each input value moved. The result is one number per time step: how much that step pushed the forecast up or down.

In [ ]:
def integrated_gradients(f, x, baseline, steps=64):
    # f: maps a batch (B, T, C) to B numbers. x, baseline: one series each,
    # shape (T, C). Returns attributions, shape (T, C).
    # the m points on the path
    alphas = (torch.arange(steps, device=x.device) + 0.5) / steps
    # ✏️ TODO 1a - all points on the path: start at the baseline and move a
    # fraction alpha of the way towards x.
    #   Replace BLANK with the full distance from the baseline to the series: x
    #   - baseline
    path = baseline + alphas.view(-1, 1, 1) * (BLANK)
    path.requires_grad_(True)
    # ✏️ TODO 1b - the gradient of the output with respect to every point on the
    # path.
    #   torch.autograd.grad(output, input) needs the input we differentiate by.
    #   Replace BLANK with it: path
    grads = torch.autograd.grad(f(path).sum(), BLANK)[0]
    # ✏️ TODO 1c - average the gradient over the points of the path, then
    # multiply by the distance moved.
    #   grads has shape (steps, T, C); the path points are its first dimension.
    #   Replace BLANK with that dimension: 0
    return ((x - baseline) * grads.mean(BLANK)).detach()

### ✅ Check: `integrated_gradients`

In [ ]:
x, zero = X_valid[0], torch.zeros_like(X_valid[0])
a = integrated_gradients(linear, x, zero)
assert (
    a.shape == x.shape
), f"attributions must have the shape of the input {tuple(x.shape)}, got {tuple(a.shape)}"
exact = linear.out.weight[0] * x[:, 0]
assert torch.allclose(
    a[:, 0], exact, atol=1e-5
), "on the linear model IG must equal weight x input"
a = integrated_gradients(lstm, x, zero, steps=256)
gap = float(lstm(x[None]) - lstm(zero[None]))
assert (
    abs(float(a.sum()) - gap) < 0.01
), f"completeness fails on the LSTM: sum {float(a.sum()):.4f} vs f(x) - f(x') {gap:.4f}"
print(
    "✅ integrated_gradients is right: exact on the linear model, complete on the LSTM"
)

## 3. Completeness and the number of steps · 🟢 core

The sum over the path only approximates an integral. How many steps does this small LSTM need before the attributions add up?

> **What this cell does.** It runs IG with more and more steps on the walk and compares the sum of the attributions with `f(x) - f(x')`.

In [ ]:
rows = []
for m in [1, 2, 4, 8, 16, 32, 64, 256]:
    a = integrated_gradients(lstm, x, zero, steps=m)
    rows.append(
        {
            "steps": m,
            "sum of attributions": float(a.sum()),
            "f(x) - f(x')": gap,
            "error": abs(float(a.sum()) - gap),
        }
    )
display(pd.DataFrame(rows).round(4))

> **What to notice.** The error shrinks as the number of steps grows. From about 32 steps the attributions add up to the change in the forecast.

## 4. Which time steps does the forecast use? · 🟢 core

**One forecast first.** The top row shows three validation series, the true next value (cross) and the LSTM forecast (dot). The bottom row shows the IG attribution of every time step: red pushes the forecast up, blue pushes it down.

> **What this cell does.** It explains three single forecasts. Top: the series, the true next value and the LSTM forecast. Bottom: the IG value of every time step.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(13, 5.2), sharex=True)
for j in range(3):
    x = X_valid[j]
    a = integrated_gradients(lstm, x, torch.zeros_like(x))[:, 0].numpy()
    axes[0, j].plot(x[:, 0], ".-", color=GREY)
    axes[0, j].plot(n_steps, y_valid[j], "kx", ms=9, label="true next value")
    axes[0, j].plot(
        n_steps, float(lstm(x[None])), "o", color=RED, label="LSTM forecast"
    )
    axes[0, j].set_title(f"validation series {j}", fontsize=10)
    axes[1, j].bar(range(n_steps), a, color=np.where(a > 0, RED, BLUE))
    axes[1, j].axhline(0, color="k", lw=0.5)
    axes[1, j].set_xlabel("time step")
axes[0, 0].legend(frameon=False, fontsize=8)
axes[0, 0].set_ylabel("series")
axes[1, 0].set_ylabel("IG attribution")
plt.tight_layout()
plt.show()

> **What to notice.** The tall bars are at the right: the most recent steps move the forecast most. Red bars push the forecast up, blue bars down.

**Now many forecasts.** One series proves little. Average the size of the attribution, $|\mathrm{IG}_i|$, over 300 validation series, for both models. Each curve is scaled to add up to 1, so it reads as *the share of the explanation that falls on each time step*.

> **What this cell does.** It averages the size of the IG values over 300 series, for both models, so we see a general pattern instead of one example.

In [ ]:
N_EXPLAIN = 300


def mean_abs_ig(model, baseline_fn=torch.zeros_like):
    A = torch.stack(
        [
            integrated_gradients(model, x, baseline_fn(x), steps=32)[:, 0].abs()
            for x in X_valid[:N_EXPLAIN]
        ]
    )
    return (A.mean(0) / A.mean(0).sum()).numpy()


profile = {name: mean_abs_ig(m) for name, m in MODELS.items()}

fig, ax = plt.subplots(figsize=(9, 3.6))
ax.plot(profile["Linear"], "o-", ms=3, color=GREY, label="Linear")
ax.plot(profile["LSTM"], "o-", ms=3, color=BLUE, label="LSTM")
ax.set_xlabel("time step (49 = the most recent value)")
ax.set_ylabel("share of |IG|")
ax.legend(frameon=False)
plt.tight_layout()
plt.show()
display(
    pd.DataFrame(
        {
            name: {
                f"share of |IG| on the last {k} steps": p[-k:].sum()
                for k in [1, 3, 5, 10, 25]
            }
            for name, p in profile.items()
        }
    ).round(2)
)

> **What to notice.** Both curves rise steeply at the right. The table says what share of the explanation falls on the last 1, 3, 5, 10, 25 steps.

## 5. The baseline matters · ⚪ if time

IG explains the forecast **relative to a baseline**. A flat line at zero is one choice of "no information". Two others are just as defensible:

| Baseline | What it means |
|---|---|
| `zero` | a flat line at 0, the long-run average of every series |
| `mean` | a flat line at this series' own average |
| `last` | a flat line at this series' last value: "nothing changed" |

Completeness holds for every baseline, but the attributions are different numbers, because the question is different.

> **What this cell does.** It explains the same forecast with three different baselines.

In [ ]:
BASELINES = {
    "zero": torch.zeros_like,
    "mean": lambda x: torch.full_like(x, float(x.mean())),
    "last": lambda x: torch.full_like(x, float(x[-1, 0])),
}
x = X_valid[0]
attr = {}
fig, axes = plt.subplots(1, 3, figsize=(13, 2.8), sharey=True)
for ax, (name, fn) in zip(axes, BASELINES.items()):
    b = fn(x)
    attr[name] = integrated_gradients(lstm, x, b)[:, 0].numpy()
    ax.bar(
        range(n_steps), attr[name], color=np.where(attr[name] > 0, RED, BLUE)
    )
    ax.axhline(0, color="k", lw=0.5)
    ax.set_title(
        f"baseline: {name}    f(x) - f(x') = {float(lstm(x[None]) - lstm(b[None])):+.2f}",
        fontsize=10,
    )
    ax.set_xlabel("time step")
axes[0].set_ylabel("IG attribution")
plt.tight_layout()
plt.show()
print("Rank correlation between the three explanations of the same forecast:")
display(
    pd.DataFrame(
        [
            [spearmanr(attr[a], attr[b])[0] for b in BASELINES]
            for a in BASELINES
        ],
        index=list(BASELINES),
        columns=list(BASELINES),
    ).round(2)
)

> **What to notice.** The three bar charts are different although the model and the series are the same. IG always answers *compared with what?*

## 6. Occlusion over time windows: the model-agnostic view · ⚪ if time

IG needs gradients, so it only works when we can open the model. **Occlusion** needs nothing but forecasts: hide a **window** of time steps (set it to the fill value), run the model again, and record how far the forecast moves.

$$\text{drop}_j = f(x) - f(x \text{ with window } j \text{ hidden})$$

This is the occlusion of the EuroSAT lab, with a window in time in place of a patch in the image.

To compare with IG, add up the IG attributions inside each window. On the **linear** model the two must agree exactly: hiding a window removes $\sum w_i x_i$ over that window, which is the IG sum.

> **The idea in words.** Occlusion needs no gradients. Hide 5 time steps (overwrite them with 0), ask the model again, and see how far the forecast moves. Do this for each of the 10 windows.

In [ ]:
@torch.no_grad()
def occlusion_time(f, x, window=5, fill=0.0):
    # Hide one window of time steps at a time. x: (T, C). Returns the drop for
    # each of the T // window windows.
    n = x.shape[0] // window
    # n copies of the series
    batch = x.repeat(n, 1, 1)
    # ✏️ TODO 2a - in copy j, hide window j: overwrite its time steps. Replace
    # BLANK with the value that hides them: fill
    for j in range(n):
        batch[j, j * window : (j + 1) * window] = BLANK
    # ✏️ TODO 2b - the drop: forecast for the full series minus the forecasts
    # for the occluded copies.
    #   f(x[None]) is the full forecast. Replace BLANK with the occluded copies
    #   built above: batch
    return (f(x[None]) - f(BLANK)).numpy()

### ✅ Check: `occlusion_time`

In [ ]:
x, zero = X_valid[0], torch.zeros_like(X_valid[0])
d = occlusion_time(linear, x, window=5)
assert d.shape == (
    10,
), f"50 steps and a window of 5 give 10 drops, got shape {d.shape}"
ig_windows = (
    integrated_gradients(linear, x, zero)[:, 0].view(10, 5).sum(1).numpy()
)
assert np.allclose(
    d, ig_windows, atol=1e-5
), "on the linear model the occlusion drop of a window must equal the IG sum over that window"
whole = occlusion_time(lstm, x, window=50)
assert np.allclose(
    whole, float(lstm(x[None]) - lstm(zero[None])), atol=1e-5
), "hiding everything must give f(x) - f(flat line at the fill value)"
print("✅ occlusion_time is right")

> **What this cell does.** It compares occlusion with IG on 200 series, window by window.

In [ ]:
W = 5
N_COMPARE = 200
occ = np.stack([occlusion_time(lstm, x, W) for x in X_valid[:N_COMPARE]])
igw = np.stack(
    [
        integrated_gradients(lstm, x, torch.zeros_like(x), steps=32)[:, 0]
        .view(-1, W)
        .sum(1)
        .numpy()
        for x in X_valid[:N_COMPARE]
    ]
)
rho = np.mean([spearmanr(o, g)[0] for o, g in zip(occ, igw)])

fig, axes = plt.subplots(1, 2, figsize=(12, 3.6))
x_pos = np.arange(n_steps // W)
axes[0].bar(
    x_pos - 0.2,
    np.abs(occ).mean(0),
    0.4,
    color=GREY,
    label="occlusion: mean |drop|",
)
axes[0].bar(
    x_pos + 0.2,
    np.abs(igw).mean(0),
    0.4,
    color=BLUE,
    label="IG: mean |sum over the window|",
)
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels(
    [f"{j * W}-{(j + 1) * W - 1}" for j in x_pos], fontsize=8
)
axes[0].set_xlabel("window of time steps")
axes[0].legend(frameon=False, fontsize=9)
axes[1].scatter(igw.ravel(), occ.ravel(), s=4, alpha=0.4, color=BLUE)
lim = np.abs(occ).max() * 1.05
axes[1].plot([-lim, lim], [-lim, lim], color=GREY, lw=1)
axes[1].set_xlabel("IG, summed over the window")
axes[1].set_ylabel("occlusion drop")
axes[1].set_title(
    f"LSTM, {N_COMPARE} series x {n_steps // W} windows", fontsize=10
)
plt.tight_layout()
plt.show()
print(f"Mean rank correlation between occlusion and IG, per series: {rho:.2f}")

> **What to notice.** The grey and blue bars have the same shape, and the points lie near the diagonal: two very different methods tell the same story.

## 7. Test the explanation: keep only the last *k* steps · 🟢 core

IG put most of the attribution on the last few steps and very little on the first half of the series (the table in section 4). It is tempting to conclude that the model **does not need** the early steps. A claim about the model can be tested **on the model**:

- **Keep the last *k* steps** and flatten everything before them to zero. If the early steps are not needed, the error stays low for a small *k*.
- **Hide the last *k* steps** and keep everything before them. If the last steps carry the forecast, the error jumps at once.

This is the deletion test of the method-comparison lab, along the time axis. Before you run it, guess: *with only the last 10 steps, how much worse is the LSTM?*

> **The idea in words.** An explanation is a claim about the model, and a claim can be tested. If only the last steps matter, the model should still work when we give it only the last *k* steps.

In [ ]:
def keep_last(X, k, fill=0.0):
    # A copy of the batch X (N, T, C) in which only the last k time steps
    # survive.
    # ✏️ TODO 3a - Xk[:, :T - k] selects every step BEFORE the last k. Replace
    # BLANK with the value that hides them: fill
    Xk = X.clone()
    Xk[:, : X.shape[1] - k] = BLANK
    return Xk


def hide_last(X, k, fill=0.0):
    # A copy of the batch X (N, T, C) in which the last k time steps are hidden.
    # ✏️ TODO 3b - Xk[:, T - k:] selects the LAST k steps. Replace BLANK with
    # the number of steps to hide: k
    Xk = X.clone()
    Xk[:, X.shape[1] - BLANK :] = fill
    return Xk

### ✅ Check: `keep_last` and `hide_last`

In [ ]:
T_ = torch.ones(2, 50, 1)
assert (
    keep_last(T_, 3).sum() == 6 and keep_last(T_, 3)[0, -3:].sum() == 3
), "keep_last(X, 3) must keep exactly the last 3 steps"
assert (
    hide_last(T_, 3).sum() == 94 and hide_last(T_, 3)[0, -3:].sum() == 0
), "hide_last(X, 3) must hide exactly the last 3 steps"
assert (
    torch.equal(keep_last(T_, 50), T_)
    and torch.equal(hide_last(T_, 0), T_)
    and T_.sum() == 100
), "k = 50 keeps everything, k = 0 hides nothing, and X itself must not change"
print("✅ keep_last and hide_last are right")

> **What this cell does.** It measures the error of both models when only the last *k* steps are kept (left) and when the last *k* steps are hidden (right).

In [ ]:
ks = [0, 1, 2, 3, 5, 10, 20, 30, 50]
keep = pd.DataFrame(
    {
        name: [mse(m, keep_last(X_valid, k)) for k in ks]
        for name, m in MODELS.items()
    },
    index=ks,
)
hide = pd.DataFrame(
    {
        name: [mse(m, hide_last(X_valid, k)) for k in ks]
        for name, m in MODELS.items()
    },
    index=ks,
)

fig, axes = plt.subplots(1, 2, figsize=(12, 3.6), sharey=True)
for ax, df, title in zip(
    axes, [keep, hide], ["Keep only the last k steps", "Hide the last k steps"]
):
    ax.plot(df.index, df["Linear"], "o-", color=GREY, label="Linear")
    ax.plot(df.index, df["LSTM"], "o-", color=BLUE, label="LSTM")
    ax.set_yscale("log")
    ax.set_xlabel("k")
    ax.set_title(title, fontsize=10)
axes[0].set_ylabel("validation MSE (log scale)")
axes[0].legend(frameon=False)
plt.tight_layout()
plt.show()
display(
    pd.concat({"keep the last k": keep, "hide the last k": hide}, axis=1).round(
        4
    )
)

> **What to notice.** Hiding even one recent step hurts a lot. But keeping only the last 10 steps is also clearly worse than keeping all 50: steps with a small IG value are still needed.

## 8. What did you find?

When this notebook was built, the numbers were (they move a little with the random seed):

| 2,000 validation series | Linear | LSTM |
|---|---|---|
| Validation MSE (naive forecast: 0.020) | 0.0034 | 0.0026 |
| Share of \|IG\| on the last 10 steps | 0.51 | 0.69 |
| Share of \|IG\| on the first 25 steps | 0.20 | 0.10 |
| MSE with only the last 10 steps | 0.020 | 0.014 |
| MSE with only the last 30 steps | 0.0087 | 0.0033 |
| MSE with the last step hidden | 0.10 | 0.15 |

For the LSTM, occlusion and IG agree with a rank correlation of 0.80. For one forecast, the IG maps with the `zero` and `mean` baselines correlate at 0.98; `zero` and `last` at 0.40.

Answer in one or two sentences each.

1. IG put about 70% of the LSTM's attribution on the last 10 steps. Is the forecast still good with only those 10 steps? How many steps does the LSTM need, and which table tells you?
2. A step with a small attribution is not the same as a step the model can do without. Explain the difference with the numbers from sections 4 and 7.
3. The earlier lab compared models by their error only. What does this lab add that the error could not tell you?
4. Section 5 gave three different explanations of the same forecast. Is one of them wrong? What would you report to a user?
5. Occlusion and IG agree closely here. Name one reason why they could disagree on another model or another dataset. Think about what a window filled with zeros looks like to the model.
6. The series is two smooth waves. Why is it reasonable that a good forecaster looks mostly at the most recent steps? What kind of series would force a model to look further back?

**Try it.**

- Swap the LSTM for a `nn.GRU` or a `nn.RNN` (the earlier lab's SimpleRNN). Does the model use the same steps?
- Train the earlier lab's 10-steps-ahead model (output size 10) and explain the forecast for step 10. Does the model now look further back?
- Change the fill value in `occlusion_time` from 0 to the series' last value. Does the agreement with IG (with the matching baseline) hold?

**Where this goes next.** *Sequence lab 2* explains the Transformer forecaster, where the data has a known answer built in. Lab D21 (*Saliency benchmark on time series*) runs a larger known-answer test for saliency maps, and D23 (*Linear forecast with readable weights*) shows the other route: a model you can read without any explanation method.